# OLLAMA server on google COLAB

This project servers as an Ollama server on google Colab and utilizes ngrok to expose the endpoint.


## Step 1: Installing ollama
It provides a simple API for creating, running, and managing models, as well as a library of pre-built models

https://ollama.com/

In [8]:
!apt-get install zstd -y
!curl -fsSL https://ollama.ai/install.sh | sh
!ls -l /usr/local/bin/ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
-rwxr-xr-x 1 root root 39107216 Jul 27 18:50 /usr/local/bin/ollama


## Step 2: Installing ngrok

### Download:

In [9]:
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz

--2026-07-30 08:05:13--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 99.83.220.108, 13.248.244.96, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12099070 (12M) [application/octet-stream]
Saving to: ‘ngrok-v3-stable-linux-amd64.tgz’

ngrok-v3-stable-lin 100%[===================>]  11.54M  15.5MB/s    in 0.7s    

2026-07-30 08:05:14 (15.5 MB/s) - ‘ngrok-v3-stable-linux-amd64.tgz’ saved [12099070/12099070]



### Extract the bin file:

In [10]:
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xvzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok
!rm ngrok-v3-stable-linux-amd64.tgz
!ls -l ngrok

--2026-07-30 08:05:14--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 99.83.220.108, 13.248.244.96, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12099070 (12M) [application/octet-stream]
Saving to: ‘ngrok-v3-stable-linux-amd64.tgz.1’

ngrok-v3-stable-lin 100%[===================>]  11.54M  14.8MB/s    in 0.8s    

2026-07-30 08:05:15 (14.8 MB/s) - ‘ngrok-v3-stable-linux-amd64.tgz.1’ saved [12099070/12099070]

ngrok
-rwxr-xr-x 1 root root 32903330 Jul 22 21:16 ngrok


### Set the auth token

a ngrok auth token must be aquired in order to use it.  

Follow the instruction below:  
https://ngrok.com/docs/getting-started/  

Then replace your AuthToken with `<NGROK_AUTH_TOKEN>` below:


In [11]:
!./ngrok authtoken 3Bfth6a9bRFdqZxN9Dqwt6SZ0lU_4Mi75Yo9BNZ4iiGamTnVP

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


## Final Step: Start ollama, pull llama model and stablish ngrok token

After pulling the selected model (in this example llama3) an endpoint will be created which can be used to access to ollama api.   
**You can also find the endpoint inside your ngrok panel: https://dashboard.ngrok.com/cloud-edge/endpoints**


In [12]:
# Start ollama server in the background
!nohup /usr/local/bin/ollama serve > ollama_serve.log 2>&1 &
!sleep 5

# Check if ollama server is responsive
!while ! curl -s http://localhost:11434 > /dev/null; do echo "Waiting for Ollama server..."; sleep 5; done
!echo "Ollama server is up!"

# Start ngrok in the background to expose the ollama server
!nohup ./ngrok http 11434 --host-header="localhost:11434" > ngrok.log 2>&1 &
!sleep 5 # Give ngrok a moment to start

# Get the ngrok public URL
import requests
import time

ngrok_api_url = "http://localhost:4040/api/tunnels"
public_url = None
for _ in range(10): # Try up to 10 times with a 2-second delay
    try:
        response = requests.get(ngrok_api_url)
        response.raise_for_status()
        tunnels = response.json().get("tunnels", [])
        if tunnels:
            for tunnel in tunnels:
                if tunnel.get("proto") == "https": # Prefer HTTPS tunnel
                    public_url = tunnel["public_url"]
                    break
            if not public_url and tunnels: # Fallback to http if no https
                public_url = tunnels[0]["public_url"]
            if public_url:
                break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)

if public_url:
    print(f"Ollama server exposed at: {public_url}")
    print("\nYou can now interact with the Ollama API using this URL.")
    print("To pull a model (e.g., llama3) using curl:")
    print(f"curl -X POST {public_url}/api/pull -d '{{\"name\": \"llama3\"}}'")
    print("To chat with a model (e.g., llama3 after pulling) using curl:")
    print(f"curl -X POST {public_url}/api/generate -d '{{\"model\": \"llama3\", \"prompt\": \"Why is the sky blue?\"}}'")
    print("Alternatively, if you have the Ollama client installed locally, set OLLAMA_HOST environment variable to this URL.")
else:
    print("Failed to get ngrok public URL. Check ngrok.log for errors.")
    !cat ngrok.log

Ollama server is up!
Ollama server exposed at: https://justine-bathyal-lorina.ngrok-free.dev

You can now interact with the Ollama API using this URL.
To pull a model (e.g., llama3) using curl:
curl -X POST https://justine-bathyal-lorina.ngrok-free.dev/api/pull -d '{"name": "llama3"}'
To chat with a model (e.g., llama3 after pulling) using curl:
curl -X POST https://justine-bathyal-lorina.ngrok-free.dev/api/generate -d '{"model": "llama3", "prompt": "Why is the sky blue?"}'
Alternatively, if you have the Ollama client installed locally, set OLLAMA_HOST environment variable to this URL.


## Step 3: Pulling and using the `Distendo/zen-pro` model

Now that the Ollama server is running and accessible via ngrok, we can pull the `Distendo/zen-pro` model and then interact with it.

In [ ]:
model_name = "Distendo/zen-pro"
pull_url = f"{public_url}/api/pull"

# Pull the model
print(f"Pulling model: {model_name} from {pull_url}")
response = requests.post(pull_url, json={
    "name": model_name,
    "stream": True # Enable streaming for live progress updates
}, stream=True)

# Print streaming response for progress
print("Pulling progress:")
for chunk in response.iter_content(chunk_size=None):
    try:
        print(chunk.decode('utf-8'), end='')
    except UnicodeDecodeError:
        # Handle cases where a chunk might not be a complete UTF-8 sequence
        print(chunk.decode('latin-1', errors='ignore'), end='')

response.raise_for_status() # Raise an exception for bad status codes
print(f"\nSuccessfully pulled model: {model_name}")

print("\n--- Interact with the model ---")
print(f"To chat with the {model_name} model using curl:")
print(f"curl -X POST {public_url}/api/generate -d '{{\"model\": \"{model_name}\", \"prompt\": \"Hello, what can you do?\"}}'")
print("\nTo get general model information (e.g., to confirm it's available):")
print(f"curl -X POST {public_url}/api/show -d '{{\"name\": \"{model_name}\"}}'")

Pulling model: Distendo/zen-pro from https://justine-bathyal-lorina.ngrok-free.dev/api/pull
Pulling progress:
{"status":"pulling manifest"}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496,"completed":356876}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496,"completed":15426445}
{"status":"pulling a3de86cd1c13","digest":"sha256:a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f","total":5225374496,"comp